# HSCIS-ESD Analysis
Hybrid Symbolic Clinical Inference System — full evaluation notebook

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np

from src.data.loader import load_dataset
from src.grading.fuzzy_grader import FuzzyGrader
from src.symbolic.pipeline import SymbolicPipeline
from src.models.model_a import run_model_a
from src.models.model_b import run_model_b
from src.models.model_c import run_model_c
from src.triage.biopsy_triage import BiopsyTriage
from src.evaluation.metrics import print_comparison_table, run_statistical_test, per_class_safety_analysis
from src.evaluation.explainability import train_final_model, compute_shap_values, plot_shap_global, plot_shap_beeswarm, extract_imodels_rules

X_clinical, X_histopath, X_all, y = load_dataset()
print(f'Dataset: {X_all.shape}, Classes: {y.value_counts().to_dict()}')

## Model Comparison (A vs B vs C)

In [ ]:
res_a = run_model_a(X_all, y)
res_b = run_model_b(X_clinical, y)
res_c = run_model_c(X_clinical, y)
print_comparison_table(res_a, res_b, res_c)

## Statistical Test: B vs C

In [ ]:
grader = FuzzyGrader()
X_b = grader.grade(X_clinical).reset_index(drop=True)
X_c = res_c['X_combined']

stat_result = run_statistical_test(X_b, X_c, y)
print(f"t-statistic: {stat_result['t_stat']}")
print(f"p-value:     {stat_result['p_value']}")
print(f"Significant: {stat_result['significant']}")
print(f"Mean improvement (C-B): {stat_result['mean_improvement']:.4f}")
print(stat_result['interpretation'])

## SHAP Analysis — Model C

In [ ]:
X_combined = res_c['X_combined']
final_model = train_final_model(X_combined, y)
shap_values = compute_shap_values(final_model, X_combined)
plot_shap_global(shap_values, X_combined, save_path='shap_global.png')
plot_shap_beeswarm(shap_values, X_combined, save_path='shap_psoriasis.png', class_idx=0)
print('SHAP plots saved.')

## imodels Rule Extraction

In [ ]:
rules = extract_imodels_rules(X_combined, y, max_rules=15)
print('Top IF-THEN rules (post-hoc, from Model C):')
for r in rules:
    print(' ', r)

## Biopsy Triage + Per-Class Safety Analysis

In [ ]:
pipeline = SymbolicPipeline('rules')
X_fuzzy = grader.grade(X_clinical).reset_index(drop=True)
X_symbolic = pipeline.transform(X_fuzzy)

final_model_c = train_final_model(res_c['X_combined'], y)
y_pred = final_model_c.predict(res_c['X_combined'])

safety_df = per_class_safety_analysis(X_symbolic, y, y_pred)
print(safety_df.to_string(index=False))

## Single Patient Reasoning Trace

In [ ]:
sample = X_clinical.iloc[0]
fuzzy_sample = grader.grade_series(sample)
trace = pipeline.explain(fuzzy_sample)

print('=== Clinical Reasoning Trace ===')
print(f'Certainty scores: {trace["certainty_scores"]}')
print(f'Conflict load:    {trace["conflict_load"]}')
print(f'Contradiction:    {trace["contradiction_severity"]}')
print(f'FSM state:        {trace["fsm_state"]}')
print('Fired rules:')
for r in trace['fired_rules']:
    print(f'  {r["id"]} ({r["disease"]}, tier {r["tier"]}): strength={r["firing_strength"]}, contrib={r["contribution"]}')

triage = BiopsyTriage()
certainty_vals = list(trace['certainty_scores'].values())
triage_result = triage.recommend(
    top_certainty=max(certainty_vals),
    conflict_load=trace['conflict_load'],
    fsm_state=trace['fsm_state'],
)
print(f'\nTriage recommendation: {triage_result}')
print(f'True label: {y.iloc[0]}')